In [55]:
import json
import networkx as nx
from itertools import combinations
from collections import defaultdict
from pathlib import Path
from pyvis.network import Network
import os, sys
from collections import Counter

import numpy as np
import pyLDAvis
import pyLDAvis.lda_model
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation



from networkx.algorithms.community import louvain_communities
from networkx.algorithms.community import greedy_modularity_communities


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.makedirs(PROJECT_ROOT / "graph_artifacts" / "visuals", exist_ok=True)
sys.path.append(str(PROJECT_ROOT))

from utils.utils import load, dump

PROCESSED_VIDEO_DATA_PATH = PROJECT_ROOT / "data" / "video_data_processed.json"
OUTVISUAL = PROJECT_ROOT / "data" / "GraphVisual.html" 
OUTPUTGRAPH = PROJECT_ROOT / "data" / "graph.graphml"

ARTIFACT_DIR = PROJECT_ROOT / "graph_artifacts"
VISUAL_DIR  = ARTIFACT_DIR / "visuals"

In [56]:
# Graph Helper Class
class graph:
    
# INIT ===========================================================================================================

    def __init__(self, Gr = None, gPath = None):
        self.G = nx.MultiDiGraph()
        self.G = Gr
        if gPath:
            try:
                self.G = nx.read_graphml(path=gPath)
            except FileNotFoundError as e:
                print(f"Tried reading graph from memory, failed. Graph doesn't exist yet.")

# VISUALIZE ======================================================================================================================

    def visualize(self, outPath=OUTVISUAL):
        # Get top nodes
        top_nodes = sorted(self.G.degree(), key=lambda x: x[1], reverse=True)[:1000]
        G_sub = self.G.subgraph([n for n, d in top_nodes])

        net = Network(height="750px", width="100%", directed=False, notebook=False)
        net.from_nx(G_sub)

        # Colour by node type
        for node in net.nodes:
            if self.G.nodes[node["id"]].get("node_type") == "celebrity":
                node["color"] = "#e74c3c"   # celebs
            elif self.G.nodes[node["id"]].get("node_type") == "brand":
                node["color"] = "#3498db"   # brands
            else:
                node["color"] = "#95a5a6"   # else
                
        for edge in net.edges:
            edge["label"] = ""
            edge["title"] = ""
            edge["width"] = 0.5
            edge["color"] = "#cccccc"

        net.set_options("""
        {
        "edges": {
            "arrows": { "to": { "enabled": false } },
            "color": { "color": "#cccccc", "opacity": 0.75 },
            "width": 0.5,
            "smooth": { "enabled": false }
        },
        "physics": {
            "forceAtlas2Based": {
            "gravitationalConstant": -50,
            "springLength": 100
            },
            "solver": "forceAtlas2Based",
            "stabilization": {
                "enabled": true,
                "iterations": 200,
                "fit": true
                }
            }
        }
        """)

        net.show(str(outPath), notebook=False)
        print(f"Graph saved to: {outPath}")

# RUN COMMUNITY DETECTION ======================================================================================================== 

    def communities(self, method="louvain", resolution=1.0, min_weight = 20):
        G_undirected = self.G.to_undirected() if self.G.is_directed() else self.G


        # Filter Edge weights
        if min_weight > 1:
                edges_to_remove = [(u, v) for u, v, d in G_undirected.edges(data=True)
                                if d.get("width", 0) < min_weight]
                G_undirected = G_undirected.copy()
                G_undirected.remove_edges_from(edges_to_remove)
                G_undirected.remove_nodes_from(list(nx.isolates(G_undirected)))
                print(f"After weight filter, Nodes: {G_undirected.number_of_nodes()}, Edges: {G_undirected.number_of_edges()}")

        if G_undirected.number_of_nodes() == 0:
            print(f"Graph is empty after min_weight={min_weight} filter — try a lower threshold.")
            return []


        if method == "louvain":
            result = louvain_communities(G_undirected, seed=42, resolution=resolution)

        elif method == "greedy":
            result = greedy_modularity_communities(G_undirected)

        else:
            #Default to louvian
            result = louvain_communities(G_undirected, seed=42, resolution=resolution)


        # Q modularity for Communities accuraccy score 
        modularity = nx.community.modularity(G_undirected, result)

        # Tag every node with its community ID
        for i, community in enumerate(result):
            for node in community:
                self.G.nodes[node]["community"] = i

        self.communities_result = result

        print(f"Communities found:  {len(result)}")
        print(f"Modularity (Q):     {modularity:.4f}")
        print(f"Largest community:  {max(len(c) for c in result)} nodes")
        print(f"Smallest community: {min(len(c) for c in result)} nodes")
        print()
        
        for i, community in enumerate(result):
            celebs = [n for n in community if self.G.nodes[n].get("node_type") == "celebrity"]
            brands = [n for n in community if self.G.nodes[n].get("node_type") == "brand"]
            print(f"  Community {i}: {len(community)} nodes | {len(celebs)} celebs | {len(brands)} brands")
            print(f"    Top members: {sorted(community, key=lambda n: self.G.nodes[n].get('mention_count', 0), reverse=True)[:5]}")

        return result
    
#  Visualize Communities =========================================================================================================

    def visualize_communities(self, communities, outPath=OUTVISUAL):
 
        COMMUNITY_PALETTE = [
            "#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6",
            "#1abc9c", "#e67e22", "#34495e", "#e91e63", "#00bcd4",
            "#8bc34a", "#ff5722", "#607d8b", "#795548", "#ffeb3b",
        ]
 
        # community MAP
        community_map = {}
        for i, community in enumerate(communities):
            for node in community:
                community_map[node] = i
 
        G_sub = self.G.subgraph(list(community_map.keys())).copy()
 
        # Remove edges that cross communities
        cross_edges = [(u, v) for u, v in G_sub.edges() if community_map.get(u) != community_map.get(v)]
        G_sub.remove_edges_from(cross_edges)
 
        net = Network(height="750px", width="100%", directed=False, notebook=False)
        net.from_nx(G_sub)
 
        for node in net.nodes:
            community_id   = community_map.get(node["id"], -1)
            node["color"]  = COMMUNITY_PALETTE[community_id % len(COMMUNITY_PALETTE)]
            node["title"]  = f"{node['id']} | Community {community_id}"
 
        for edge in net.edges:
            edge["label"] = ""
            edge["title"] = ""
            edge["width"] = 0.5
            edge["color"] = "#cccccc"
 
        net.set_options("""
        {
          "edges": {
            "arrows": { "to": { "enabled": false } },
            "color": { "color": "#cccccc", "opacity": 0.6 },
            "width": 0.5,
            "smooth": { "enabled": false }
          },
          "physics": {
            "forceAtlas2Based": {
              "gravitationalConstant": -50,
              "springLength": 100
            },
            "solver": "forceAtlas2Based",
            "stabilization": { "enabled": true, "iterations": 200, "fit": true }
          }
        }
        """)
 
        net.show(str(outPath), notebook=False)
        print(f"Community graph saved to: {outPath}")

# Degree centrality ==============================================================================================================

    def degree_centrality(self, outPath=ARTIFACT_DIR / "degree_centrality.json"):
        scores = nx.degree_centrality(self.G)
        sorted_scores = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

        for i, (node, score) in enumerate(list(sorted_scores.items())[:20], start=1):
            mention_count = self.G.nodes[node].get("mention_count", 0)
            node_type = self.G.nodes[node].get("node_type", "unknown")
            print(f"[{i:02}] {node:<30} {score:.4f}  |  mentions: {mention_count}  |  type: {node_type}")

        dump(sorted_scores, outPath=outPath)
        return sorted_scores

# betweenness centrality ==========================================================================================================

    def betweenness_centrality(self, outPath=ARTIFACT_DIR / "betweenness_centrality.json"):
        scores = nx.betweenness_centrality(self.G,  weight="co_mention_count")
        sorted_scores = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

        for i, (node, score) in enumerate(list(sorted_scores.items())[:20], start=1):
            mention_count = self.G.nodes[node].get("mention_count", 0)
            node_type = self.G.nodes[node].get("node_type", "unknown")
            print(f"[{i:02}] {node:<30} {score:.4f}  |  mentions: {mention_count}  |  type: {node_type}")

        dump(sorted_scores, outPath=outPath)
        return sorted_scores
    
# closeness centrality ===========================================================================================================

    def closeness_centrality(self, outPath=ARTIFACT_DIR / "closeness_centrality.json"):
        scores = nx.closeness_centrality(self.G)
        sorted_scores = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

        for i, (node, score) in enumerate(list(sorted_scores.items())[:20], start=1):
            mention_count = self.G.nodes[node].get("mention_count", 0)
            node_type = self.G.nodes[node].get("node_type", "unknown")
            print(f"[{i:02}] {node:<30} {score:.4f}  |  mentions: {mention_count}  |  type: {node_type}")

        dump(sorted_scores, outPath=outPath)
        return sorted_scores
    
# Eigenvector centrality ===========================================================================================================

    def eigenvector_centrality(self, outPath=ARTIFACT_DIR / "eigenvector_centrality.json"):
        try:
            scores = nx.eigenvector_centrality(self.G, weight="co_mention_count", max_iter=1000)
        except nx.PowerIterationFailedConvergence:
            print("Eigenvector centrality failed to converge, trying with increased iterations...")
            scores = nx.eigenvector_centrality_numpy(self.G, weight="co_mention_count")
            
        sorted_scores = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

        for i, (node, score) in enumerate(list(sorted_scores.items())[:20], start=1):
            mention_count = self.G.nodes[node].get("mention_count", 0)
            node_type = self.G.nodes[node].get("node_type", "unknown")
            print(f"[{i:02}] {node:<30} {score:.4f}  |  mentions: {mention_count}  |  type: {node_type}")

        dump(sorted_scores, outPath=outPath)
        return sorted_scores
    
# SUMMARY ========================================================================================================================

    def summary(self, outPath=ARTIFACT_DIR / "summary.json"):
        stats = {
            "nodes":            self.G.number_of_nodes(),
            "edges":            self.G.number_of_edges(),
            "density":          round(nx.density(self.G), 6),
            "is_directed":      self.G.is_directed(),
            "is_connected":     nx.is_connected(self.G) if not self.G.is_directed() else nx.is_weakly_connected(self.G),
            "avg_degree":       round(sum(d for _, d in self.G.degree()) / self.G.number_of_nodes(), 4),
            "node_types":       dict(Counter(d.get("node_type", "unknown") for _, d in self.G.nodes(data=True))),
        }

        print("GRAPH SUMMARY")
        print("=" * 50)
        for k, v in stats.items():
            print(f"  {k:<25} {v}")

        dump(stats, outPath=outPath)
        return stats

# EXPORT =========================================================================================================================

    def export(self, outPath = OUTPUTGRAPH):
        nx.write_graphml(self.G, outPath)

In [57]:
class ldaTopics:
    # Parameters ------------------------------------------------------------------------------------------

    def __init__(self):

        self.NUM_TOPICS = 5
        self.WORDS_PER_TOPIC = 20
        self.FEATURE_NUM = 1000
        self.ITERATIONS = 100

    # LDA --------------------------------------------------------------------------------------------------

    def buildCorpus(self, comments: list):
        """
        Returns list of comment corpus
        @params list: List of comments object
        returns list of corpus
        """
        self.corpus = []
        for comment in comments:
            self.corpus.append(" ".join(comment))
        return self.corpus

    def display_topics(self, model, featureNames, numTopWords):
        """
        Prints out the most associated words for each topic.

        @param model: lda model.
        @param featureNames: list of strings, representing the list of features/words.
        @param numTopWords: number of words to print per topic.
        """

        # print out the topic distributions
        for topicId, lTopicDist in enumerate(model.components_):
            print('Topic %d:' % (topicId + 1))
            print(' '.join([featureNames[i] for i in lTopicDist.argsort()[:-numTopWords - 1:-1]]))

    def ldaAnalysis(self, comments: list, outPath: Path):

        """
        Does LDA topic modeling 

        @params tokens: list of tokens

        outputs visualisation to .html file
        
        """


        print(f"Building corpus...")

        corpus = self.buildCorpus(comments)

        print(f"Built corpus from {len(comments)} comments.")

        # Counter Vectorizer

        print(f"fitting counter vectorizer")

        tfVectorizer = CountVectorizer(max_df=0.90, min_df=5, max_features=self.FEATURE_NUM, stop_words='english')
        tf = tfVectorizer.fit_transform(corpus)
        # extract the names of the features (in our case, the words)
        tfFeatureNames = tfVectorizer.get_feature_names_out()

        print(f"Done fitting CountVectorizer")

        print(f"Fitting LDA Model.")

        ldaModel = LatentDirichletAllocation(n_components=self.NUM_TOPICS, max_iter=self.ITERATIONS, learning_method='online').fit(tf)

        print(f"Done!")

        # Display Topics 

        print("Displaying TOPICS...")
        self.display_topics(ldaModel, tfFeatureNames, self.WORDS_PER_TOPIC)

        # Visualize and save Topic visual

        print("Visualizing Topics")
        panel = pyLDAvis.lda_model.prepare(
            ldaModel,
            tf,
            tfVectorizer,
            mds="mmds",
            n_jobs=1 
        )

        pyLDAvis.save_html(panel, str(outPath))

        print(f"Saved visual successfully!")

        return True







In [58]:
def co_mention_comments(comments, celeb_list, target=1000, max_stepdowns=None):
    collected_ids  = set()
    matched_combos = []

    num_celebs = len(celeb_list)
    floor = num_celebs - max_stepdowns if max_stepdowns is not None else 2


    while num_celebs >= floor and len(collected_ids) < target:
        combos     = list(combinations(celeb_list, num_celebs))
        newly_found = 0

        if num_celebs == 1:
            break
        
        for combo in combos:
            combo_set  = set(combo)

            for comment in comments:
                comment_celebs = set(comment.get("celebs", []))
                comment_id     = comment.get("comment_id")

                if comment_id in collected_ids:
                    continue
                if combo_set.issubset(comment_celebs):
                    collected_ids.add(comment_id)
                    newly_found += 1

                if len(collected_ids) >= target:
                    break

            if len(collected_ids) >= target:
                break


        print(f"  num_celebs={num_celebs} ({len(combos)} combos) found {newly_found} new comments | total: {len(collected_ids)}")
        matched_combos.append({"n_celebs": num_celebs, "newly_found": newly_found})
        num_celebs -= 1

    print(f"\nDone. Total comment IDs collected: {len(collected_ids)}")
    return list(collected_ids), matched_combos


In [59]:
# Get Data
with open(PROCESSED_VIDEO_DATA_PATH, "r", encoding="utf-8") as f:
    processed = json.load(f)

comments = processed["comments"]
entity_counts = processed["entity_counts"]

In [60]:
co_mentions = defaultdict(int)
entity_sentiment = defaultdict(list)

# GET CELEBRITY CO-MENTIONS 
for comment in comments:
    entities = list(set(comment.get("celebs", [])))

    if len(entities) < 2:
        continue

    for pair in combinations(sorted(entities), 2):
        co_mentions[pair] += 1

G1 = nx.Graph()
print(f"Building Graph 1 Celebrity co-mentions:")
print("-"*82)
# Add nodes
for celeb, count in entity_counts["celebs"].items():
    G1.add_node(celeb, node_type="celebrity", mention_count=count)

# Add edges
MIN_CO_MENTIONS = 15
for (e1, e2), weight in co_mentions.items():
    if weight >= MIN_CO_MENTIONS:
        G1.add_edge(e1, e2, weight=weight)

# No co-mentiones with anyone
G1.remove_nodes_from(list(nx.isolates(G1)))

print(f"Nodes:          {G1.number_of_nodes()}")
print(f"Edges:          {G1.number_of_edges()}")
print(f"Celebrities:    {sum(1 for _, d in G1.nodes(data=True) if d.get('node_type') == 'celebrity')}")
print("-"*82)


Building Graph 1 Celebrity co-mentions:
----------------------------------------------------------------------------------
Nodes:          34
Edges:          64
Celebrities:    34
----------------------------------------------------------------------------------


In [61]:
co_mentions = defaultdict(int)

print("Building Graph 2 Celebrity - Brand co-mention")
print("-"*82)

for comment in comments:
    celebs = list(set(comment.get("celebs", [])))
    brands = list(set(comment.get("brands", [])))

    for celeb in celebs:
        for brand in brands:
            co_mentions[(celeb, brand)] += 1

G2 = nx.Graph()

for celeb, count in entity_counts["celebs"].items():
    G2.add_node(celeb, node_type="celebrity", mention_count=count)

for brand, count in entity_counts["brands"].items():
    G2.add_node(brand, node_type="brand", mention_count=count)

MIN_CO_MENTIONS = 2
for (celeb, brand), weight in co_mentions.items():
    if weight >= MIN_CO_MENTIONS:
        G2.add_edge(celeb, brand, weight=weight)

G2.remove_nodes_from(list(nx.isolates(G2)))

print(f"Nodes:       {G2.number_of_nodes()}")
print(f"Edges:       {G2.number_of_edges()}")
print(f"Celebrities: {sum(1 for _, d in G2.nodes(data=True) if d.get('node_type') == 'celebrity')}")
print(f"Brands:      {sum(1 for _, d in G2.nodes(data=True) if d.get('node_type') == 'brand')}")
print("-"*82)

Building Graph 2 Celebrity - Brand co-mention
----------------------------------------------------------------------------------
Nodes:       83
Edges:       120
Celebrities: 51
Brands:      32
----------------------------------------------------------------------------------


In [62]:
CG = graph(G1)

# EXPORT CELEB GRAPH
celebVisual = VISUAL_DIR / "celebs.html"
celebGraph = ARTIFACT_DIR / "celebs.graphml"
CG.visualize(outPath=celebVisual)
CG.export(outPath=celebGraph)

c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\visuals\celebs.html
Graph saved to: c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\visuals\celebs.html


In [63]:
CBG = graph(G2)

# EXPORT CELEB - BRAND GRAPH 
celebBrandVisual = VISUAL_DIR / "CBG.html"
celebBrandGraph = ARTIFACT_DIR / "CBG.graphml"
CBG.visualize(outPath=celebBrandVisual)
CBG.export(outPath=celebBrandGraph)


c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\visuals\CBG.html
Graph saved to: c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\visuals\CBG.html


In [64]:
print(f"Calculating degree centrality on Celebrity Graph")
print("-"*89)
CG.degree_centrality()
print("-"*89)

print(f"Calculating betweenness centrality on Celebrity Graph")
print("-"*89)
CG.betweenness_centrality()
print("-"*89)

print(f"Calculating closeness centrality on Celebrity Graph")
print("-"*89)
CG.closeness_centrality()
print("-"*89)

print(f"Calculating eigenvector centrality on Celebrity Graph")
print("-"*89)
CG.eigenvector_centrality()
print("-"*89)

Calculating degree centrality on Celebrity Graph
-----------------------------------------------------------------------------------------
[01] Beyonce                        0.6667  |  mentions: 1335  |  type: celebrity
[02] LISA                           0.3030  |  mentions: 1044  |  type: celebrity
[03] Emma Chamberlain               0.2121  |  mentions: 372  |  type: celebrity
[04] Rihanna                        0.1818  |  mentions: 739  |  type: celebrity
[05] Jisoo                          0.1818  |  mentions: 1284  |  type: celebrity
[06] Rose                           0.1818  |  mentions: 935  |  type: celebrity
[07] Madonna                        0.1818  |  mentions: 432  |  type: celebrity
[08] Anok Yai                       0.1515  |  mentions: 155  |  type: celebrity
[09] JENNIE                         0.1515  |  mentions: 411  |  type: celebrity
[10] Sabrina Carpenter              0.1515  |  mentions: 200  |  type: celebrity
[11] Ningning                       0.1515  |  m

In [65]:
CG.summary()

GRAPH SUMMARY
  nodes                     34
  edges                     64
  density                   0.114082
  is_directed               False
  is_connected              False
  avg_degree                3.7647
  node_types                {'celebrity': 34}


{'nodes': 34,
 'edges': 64,
 'density': 0.114082,
 'is_directed': False,
 'is_connected': False,
 'avg_degree': 3.7647,
 'node_types': {'celebrity': 34}}

In [66]:
print(f"Calculating degree centrality on Brand Graph")
print("-"*89)
CBG.degree_centrality()
print("-"*89)

print(f"Calculating betweenness centrality on Brand Graph")
print("-"*89)
CBG.betweenness_centrality()
print("-"*89)

print(f"Calculating closeness centrality on Brand Graph")
print("-"*89)
CBG.closeness_centrality()
print("-"*89)

print(f"Calculating eigenvector centrality on Brand Graph")
print("-"*89)
CBG.eigenvector_centrality()
print("-"*89)


Calculating degree centrality on Brand Graph
-----------------------------------------------------------------------------------------
[01] Dior                           0.1951  |  mentions: 87  |  type: brand
[02] Saint Laurent                  0.1829  |  mentions: 256  |  type: brand
[03] Robert Wun                     0.1707  |  mentions: 161  |  type: brand
[04] Chanel                         0.1220  |  mentions: 71  |  type: brand
[05] LISA                           0.1098  |  mentions: 1044  |  type: celebrity
[06] Mugler                         0.1098  |  mentions: 77  |  type: brand
[07] JENNIE                         0.0976  |  mentions: 411  |  type: celebrity
[08] Jisoo                          0.0854  |  mentions: 1284  |  type: celebrity
[09] Balenciaga                     0.0854  |  mentions: 44  |  type: brand
[10] Beyonce                        0.0732  |  mentions: 1335  |  type: celebrity
[11] Rose                           0.0732  |  mentions: 935  |  type: celebrity

In [67]:
CBG.summary()

GRAPH SUMMARY
  nodes                     83
  edges                     120
  density                   0.035263
  is_directed               False
  is_connected              False
  avg_degree                2.8916
  node_types                {'celebrity': 51, 'brand': 32}


{'nodes': 83,
 'edges': 120,
 'density': 0.035263,
 'is_directed': False,
 'is_connected': False,
 'avg_degree': 2.8916,
 'node_types': {'celebrity': 51, 'brand': 32}}

In [68]:
celebCommunities = CG.communities(method="louvian", min_weight=32)
communities_serializable = [list(community) for community in celebCommunities]
dump(communities_serializable, outPath=ARTIFACT_DIR / "celebCommunities.json")

print(f"celebCommnities: {celebCommunities[:10]}")

After weight filter, Nodes: 15, Edges: 18
Communities found:  3
Modularity (Q):     0.4707
Largest community:  7 nodes
Smallest community: 2 nodes

  Community 0: 7 nodes | 7 celebs | 0 brands
    Top members: ['Beyonce', 'Rihanna', 'Madonna', 'Emma Chamberlain', 'Sabrina Carpenter']
  Community 1: 6 nodes | 6 celebs | 0 brands
    Top members: ['Jisoo', 'LISA', 'Rose', 'JENNIE', 'Ningning']
  Community 2: 2 nodes | 2 celebs | 0 brands
    Top members: ['Karan Johar', 'Isha Ambani']
celebCommnities: [{'Emma Chamberlain', 'Blue Ivy', 'Sabrina Carpenter', 'Jay-Z', 'Beyonce', 'Rihanna', 'Madonna'}, {'LISA', 'Rose', 'Jisoo', 'JENNIE', 'Ningning', 'Karina'}, {'Isha Ambani', 'Karan Johar'}]


In [69]:
brandCommunities = CBG.communities(method="greedy", min_weight=5)
brand_communities_serializable = [list(community) for community in celebCommunities]
dump(brand_communities_serializable, outPath=ARTIFACT_DIR / "brandCommunities.json")

print(f"celebCommnities: {brandCommunities[:10]}")

After weight filter, Nodes: 30, Edges: 30
Communities found:  10
Modularity (Q):     0.5267
Largest community:  7 nodes
Smallest community: 2 nodes

  Community 0: 7 nodes | 5 celebs | 2 brands
    Top members: ['Jisoo', 'Rose', 'JENNIE', 'Dior', 'Chanel']
  Community 1: 5 nodes | 4 celebs | 1 brands
    Top members: ['Heidi Klum', 'Sabrina Carpenter', 'Robert Wun', 'Audrey Nuna', 'Naomi Osaka']
  Community 2: 4 nodes | 3 celebs | 1 brands
    Top members: ['Madonna', 'Saint Laurent', 'Connor Storrie', 'Anthony Vaccarello']
  Community 3: 2 nodes | 1 celebs | 1 brands
    Top members: ['Beyonce', 'Balenciaga']
  Community 4: 2 nodes | 1 celebs | 1 brands
    Top members: ['LISA', 'Bulgari']
  Community 5: 2 nodes | 1 celebs | 1 brands
    Top members: ['Chloe', 'Chloe Malle']
  Community 6: 2 nodes | 1 celebs | 1 brands
    Top members: ['Emma Chamberlain', 'Mugler']
  Community 7: 2 nodes | 1 celebs | 1 brands
    Top members: ['Kylie Jenner', 'Skims']
  Community 8: 2 nodes | 1 celeb

In [70]:
celebCommunityVis = ARTIFACT_DIR / "celebCommunity.html"
CG.visualize_communities(communities=celebCommunities, outPath=celebCommunityVis)

c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\celebCommunity.html
Community graph saved to: c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\celebCommunity.html


In [ ]:
brandCommunityVis = ARTIFACT_DIR / "brandCommunity.html"
CBG.visualize_communities(communities=brandCommunities, outPath=celebCommunityVis)

c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\celebCommunity.html
Community graph saved to: c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\celebCommunity.html


In [71]:
# get community comments.
community_comment_list = []

for celebs in celebCommunities:
    comment_ids, matches = co_mention_comments(comments=comments, celeb_list=celebs, target= 1000, max_stepdowns=2)
    community_comment_list.append(comment_ids)



  num_celebs=7 (1 combos) found 0 new comments | total: 0
  num_celebs=6 (7 combos) found 2 new comments | total: 2
  num_celebs=5 (21 combos) found 4 new comments | total: 6

Done. Total comment IDs collected: 6
  num_celebs=6 (1 combos) found 11 new comments | total: 11
  num_celebs=5 (6 combos) found 6 new comments | total: 17
  num_celebs=4 (15 combos) found 114 new comments | total: 131

Done. Total comment IDs collected: 131
  num_celebs=2 (1 combos) found 32 new comments | total: 32

Done. Total comment IDs collected: 32


In [72]:
#map CID to comment tokens
comment_map = {c["comment_id"]: c["comment_tokens_topic"] for c in comments}
lda = ldaTopics()
for i, comment_ids in enumerate(community_comment_list, start=1):

    print(f"COMMUNITY {i}")
    print("-"*82)

    community_tokens = [comment_map[cid] for cid in comment_ids if cid in comment_map]
    
    outPath = VISUAL_DIR / f"lda_community_{i}.html"
    lda.ldaAnalysis(community_tokens, outPath=outPath)


COMMUNITY 1
----------------------------------------------------------------------------------
Building corpus...
Built corpus from 6 comments.
fitting counter vectorizer
Done fitting CountVectorizer
Fitting LDA Model.
Done!
Displaying TOPICS...
Topic 1:
tyla mcrae tate cat doja ciara teyana taylor ivy blue xcx yseult
Topic 2:
ciara doja taylor blue cat teyana mcrae tyla ivy xcx tate yseult
Topic 3:
doja xcx ivy tyla cat taylor blue yseult mcrae ciara teyana tate
Topic 4:
taylor cat yseult teyana ciara doja mcrae blue xcx ivy tyla tate
Topic 5:
yseult ivy mcrae ciara tate tyla xcx teyana blue taylor doja cat
Visualizing Topics


c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:744: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9. To suppress this warning, provide some value of `n_init`.
  warnings.warn(
c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:754: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:771: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(


Saved visual successfully!
COMMUNITY 2
----------------------------------------------------------------------------------
Building corpus...
Built corpus from 131 comments.
fitting counter vectorizer
Done fitting CountVectorizer
Fitting LDA Model.
Done!
Displaying TOPICS...
Topic 1:
ningning outfits best karina ate plain blink dresses looked boring nailed pink outfit better time event mean girl know ysl
Topic 2:
like dress met outfit gala art theme wore good look slayed gown really ate year boring inspired people better members
Topic 3:
love beautiful blackpink looking stunning girls blink look amazing pretty year know giving mermaid people debut slayed looked hard members
Topic 4:
look pretty looks time theme met red gala carpet comes giving new fashion good dress creative looking blue ate year
Topic 5:
interview blackpink blinks members better come looked met ate year think stunning wore blink nailed make literally looks gala hard
Visualizing Topics


c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:744: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9. To suppress this warning, provide some value of `n_init`.
  warnings.warn(
c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:754: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:771: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(


Saved visual successfully!
COMMUNITY 3
----------------------------------------------------------------------------------
Building corpus...
Built corpus from 32 comments.
fitting counter vectorizer
Done fitting CountVectorizer
Fitting LDA Model.
Done!
Displaying TOPICS...
Topic 1:
art gala indian manish malhotra fashion met theme jaipur indians looks carpet look natasha patel mona ananya sudha birla reddy
Topic 2:
manish malhotra ananya look birla sudha jaipur outfits reddy art patel mona natasha theme indian looks south culture indians gala
Topic 3:
looks malhotra manish indian theme culture patel mona ananya met gala birla south look jaipur indians sudha reddy natasha fashion
Topic 4:
indian art outfits ananya birla met fashion gala indians look manish malhotra south theme culture carpet natasha reddy sudha mona
Topic 5:
reddy malhotra manish sudha birla ananya natasha mona patel indians carpet south jaipur met gala indian theme looks look outfits
Visualizing Topics
Saved visual suc

c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:744: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9. To suppress this warning, provide some value of `n_init`.
  warnings.warn(
c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:754: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:771: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(
